<a href="https://colab.research.google.com/github/hamzafarooq/multi-agent-course/blob/main/modules/Module_3_Production_Agentic_RAG_AI_Systems/003.%20Agentic%20Router_semantic_caching_rbac.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic RAG with Semantic Cache

This notebook combines three concepts:

- **Semantic Cache** — A FAISS-backed cache that stores previous query embeddings and their answers. When a new query is semantically similar to a cached one, the stored answer is returned instantly — no LLM or API call needed.
- **Agentic RAG** — An intelligent retrieval system that routes queries to the right knowledge source: OpenAI documentation (via Qdrant), 10-K financial filings (via Qdrant), or live internet search (via SerpApi).
- **Role-Based Access Control (RBAC)** *(optional)* — An access-control layer that checks a user's role against file-level permissions **before** a query is allowed to reach the cache or the retrieval pipeline at all. Unauthorized requests are rejected immediately, with no embedding, retrieval, or LLM call.

## Architecture

```
User Query (+ user_id, file_id)
    │
    ▼
┌─────────────────────────────┐
│   RBAC Check (optional)     │  ──DENIED──▶  🚫 Reject (no data touched)
│  role → file permission?    │
└──────────────┬──────────────┘
               │ ALLOWED / not used
               ▼
┌─────────────────────────────┐
│  Is query time-sensitive?   │  ──YES──▶  Agentic RAG (no caching)
│  (current events, "today",  │
│   live data, etc.)          │
└──────────────┬──────────────┘
               │ NO
               ▼
┌─────────────────────────────┐
│   Semantic Cache Lookup     │  ──HIT──▶  Return cached answer ⚡
│   (FAISS similarity search) │
└──────────────┬──────────────┘
               │ MISS
               ▼
┌─────────────────────────────┐
│      Agentic RAG Router     │
│   (GPT-5.6 classifies query) │
└──────┬──────────┬───────────┘
       │          │           │
  OPENAI      10K_DOC    INTERNET
  QUERY       QUERY       QUERY
    │            │            │
  Qdrant      Qdrant      SerpApi
  (RAG)       (RAG)      (live web)
       │          │           │
       └──────────┴───────────┘
               │
               ▼
    Store answer in cache 💾
               │
               ▼
          Return answer
```

*The RBAC gate is optional and only applies when a request is scoped to a specific file (Section 6's `secure_agentic_rag()`). The unrestricted `agentic_rag_with_cache()` used in Section 5 skips straight to the time-sensitivity check, as before.*

## Why combine them?

- **Speed**: Cached answers return in milliseconds vs. 2–5 seconds for full RAG.
- **Cost**: Fewer LLM and API calls for repeated or similar questions.
- **Correctness**: Time-sensitive queries (e.g., *"What happened today?"*) always bypass the cache to ensure fresh answers.
- **Security**: With RBAC enabled, access checks happen before any retrieval or LLM call, so unauthorized requests never reach the underlying data or incur API cost.


## 1. Setup

Install dependencies, clone the course repository (which contains `rag_helpers.py` and the pre-built Qdrant vector database), and import the helper module.

In [ ]:
# Dependencies. Safe to skip on a re-run if the environment is already set up —
# the setup cell below is separate on purpose, so skipping this one costs you nothing.
!pip install -q -U pip setuptools
!pip install -q "transformers==4.48.0" "sentence-transformers==3.4.1" "einops==0.8.1" faiss-cpu openai qdrant_client python-dotenv nest_asyncio

In [ ]:
# Setup — always run this one.
import os, sys, shutil, nest_asyncio

# The Nomic model ships remote code; a stale copy in the HF cache causes confusing
# load errors, so clear it and let it re-download.
_nomic_cache = os.path.expanduser("~/.cache/huggingface/modules/transformers_modules/nomic-ai")
if os.path.exists(_nomic_cache):
    shutil.rmtree(_nomic_cache)
    print("Cleared cached Nomic remote code — it will be re-downloaded fresh.")

try:
    import google.colab
    _REPO = "/content/multi-agent-course"
    if not os.path.exists(_REPO):
        os.system("git clone https://github.com/hamzafarooq/multi-agent-course.git")
        print("Repository cloned ✅")
    else:
        print("Repository already present ✅")
    _MODULE_DIR = f"{_REPO}/modules/Module_3_Production_Agentic_RAG_AI_Systems"
except ImportError:
    # Running locally — rag_helpers.py sits next to this notebook
    _MODULE_DIR = os.getcwd()
    print(f"Running locally — helpers path: {_MODULE_DIR}")

sys.path.insert(0, _MODULE_DIR)
nest_asyncio.apply()  # Required for asyncio.run() inside Jupyter/Colab

In [2]:
from rag_helpers import init_rag, SemanticCaching, agentic_rag_with_cache
print("✅ Helpers imported from rag_helpers.py")

✅ Helpers imported from rag_helpers.py


## 2. API Keys

**On Google Colab** — store keys in the Secrets panel (`🔑` icon, left sidebar):
| Secret name | Where to get it |
|---|---|
| `SERP_API_KEY` | [serpapi.com](https://serpapi.com) |
| `OPENAI_API_KEY` | [platform.openai.com](https://platform.openai.com) |

**Running locally** — add keys to `Module_3_Production_Agentic_RAG_AI_Systems/.env`:
```
serp_api_key=<your_key>
openai_api_key=<your_key>
```
The cell below detects the environment automatically and loads from the right source.

In [3]:
# ── Load API keys ─────────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    serp_api_key   = userdata.get('SERP_API_KEY')
    openai_api_key = userdata.get('OPENAI_API_KEY')
    QDRANT_PATH    = f"{_REPO}/modules/Module_3_Production_Agentic_RAG_AI_Systems/Agentic_RAG/qdrant_data"
    print("Colab: credentials loaded from Secrets.")
except ImportError:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv())
    serp_api_key   = os.getenv("serp_api_key") or os.getenv("SERP_API_KEY") or os.getenv("SERPAPI_KEY")
    openai_api_key = os.getenv("openai_api_key") or os.getenv("OPENAI_API_KEY")
    QDRANT_PATH    = os.path.join(_MODULE_DIR, "Agentic_RAG", "qdrant_data")
    print("Local: credentials loaded from .env.")

print(f"SerpApi key:    {'✅' if serp_api_key else '❌ MISSING'}")
print(f"OpenAI API key: {'✅' if openai_api_key else '❌ MISSING'}")

# ── Initialise the RAG pipeline (loads models + connects to Qdrant) ───────────
init_rag(openai_api_key=openai_api_key, serp_api_key=serp_api_key, qdrant_path=QDRANT_PATH)

Colab: credentials loaded from Secrets.
SerpApi key:    ✅
OpenAI API key: ✅
Loading Nomic text model for Qdrant retrieval embeddings...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/nomic-ai/nomic-bert-2048:
- configuration_hf_nomic_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_hf_nomic_bert.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/nomic-ai/nomic-bert-2048:
- modeling_hf_nomic_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/547M [00:00<?, ?B/s]

✅ RAG pipeline ready.


## 3. Create the Semantic Cache

`SemanticCaching` is defined in `rag_helpers.py`. It provides:

| Method | Purpose |
|---|---|
| `is_time_sensitive(q)` | Returns `True` for questions with temporal keywords — these always bypass the cache |
| `check_cache(q)` | Embeds the query and runs a FAISS nearest-neighbour search; returns hit/miss + pre-computed embedding |
| `add_to_cache(q, answer, embedding)` | Persists a new entry to FAISS + JSON after a RAG call |

**Similarity threshold** (`threshold=0.2`): distance ≤ 0.2 (Euclidean) counts as a hit. Lower = stricter matching. Try `0.1` for exact-ish matches or `0.35` for a looser hit rate.

In [4]:
# Instantiate the semantic cache
# Set clear_on_init=True to wipe any previously stored entries
cache = SemanticCaching(json_file='rag_cache.json', threshold=0.2, clear_on_init=True)

Loading Nomic embedding model for semantic cache...


modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

Cache embedding model ready.
Semantic cache cleared.


## 4. Agentic RAG Pipeline (from `rag_helpers.py`)

The pipeline components are all defined in `rag_helpers.py` — see the file for full implementations.

| Function | What it does |
|---|---|
| `get_internet_content(query)` | Live Google search via SerpApi |
| `route_query(query)` | GPT-4o classifies into `OPENAI_QUERY`, `10K_DOCUMENT_QUERY`, or `INTERNET_QUERY` |
| `_retrieve_and_respond(query, action)` | Embeds query → searches the right Qdrant collection → generates a cited RAG answer |
| `_run_rag_pipeline(query)` | Orchestrates routing + handler dispatch; returns the answer string |
| **`agentic_rag_with_cache(query, cache)`** | **Main entry point** — applies the cache layer on top of the full pipeline |

**Qdrant collections loaded by `init_rag()`:**
- `opnai_data` — OpenAI Agents documentation  
- `10k_data`   — Uber 2021 & Lyft 2024 10-K filings

## 5. Demo — Semantic Cache + Agentic RAG in Action

`agentic_rag_with_cache(query, cache)` is the single function to call. It handles all routing, retrieval, caching, and display automatically.

### Test queries — three cache paths

| Query | Expected path |
|---|---|
| *"What was Uber's revenue in 2021?"* | Cache MISS → 10K RAG → stored |
| *"How much did Uber earn in 2021?"* | Cache HIT (semantically similar) |
| *"How do I build an agent with the OpenAI Agents SDK?"* | Cache MISS → OpenAI RAG → stored |
| *"What are the best AI tools this week?"* | Time-sensitive → bypass cache → SerpApi |
| *"What is the current stock price of Apple?"* | Time-sensitive → bypass cache → SerpApi |
| *"What are the most popular open-source LLMs?"* | Cache MISS → INTERNET → SerpApi → stored |

In [5]:
result = agentic_rag_with_cache("What was Uber's revenue in 2021?", cache)

👤 Query: What was Uber's revenue in 2021?

❌ Cache MISS — running Agentic RAG pipeline...

📍 Route: 10K_DOCUMENT_QUERY  |  Asks about Uber's company financials.

💾 Cached for future similar queries.

🤖 Response:
Uber's revenue in 2021 was **$3,208,323,000**, or approximately **$3.21 billion**.[1]



In [6]:
# Test 2: Cache HIT — semantically similar to Test 1, returns instantly from cache
result = agentic_rag_with_cache("How much did Uber earn in fiscal year 2021?", cache)

👤 Query: How much did Uber earn in fiscal year 2021?

✅ Cache HIT (row 0, similarity: 0.838, 0.131s)

🤖 Response (cached):
Uber's revenue in 2021 was **$3,208,323,000**, or approximately **$3.21 billion**.[1]



In [7]:
# Test 3: Cache MISS — routes to OPENAI_QUERY and stores result
result = agentic_rag_with_cache("How do I build an agent with the OpenAI Agents SDK?", cache)

👤 Query: How do I build an agent with the OpenAI Agents SDK?

❌ Cache MISS — running Agentic RAG pipeline...

📍 Route: OPENAI_QUERY  |  Asks about building an agent using the OpenAI Agents SDK.

💾 Cached for future similar queries.

🤖 Response:
An agent in the OpenAI Agents SDK is built from three core parts:

1. **Model** — the LLM that reasons and makes decisions  
2. **Tools** — functions or APIs the agent can call  
3. **Instructions** — the behavior and constraints that guide it [1]

### 1. Install and configure the SDK

```bash
pip install openai-agents
```

Set your API key:

```bash
export OPENAI_API_KEY="your-api-key"
```

### 2. Define a tool

Tools should perform a specific external action. For example:

```python
from agents import function_tool

@function_tool
def get_weather(city: str) -> str:
    """Return the current weather for a city."""
    # Replace this with a real weather API call.
    return f"The weather in {city} is sunny and 22°C."
```

### 3. Create the agent

In [8]:
# Test 4: Time-sensitive query — BYPASSES cache, calls Ares API for live answer
result = agentic_rag_with_cache("What are the best AI tools this week?", cache)

👤 Query: What are the best AI tools this week?

⏰ Time-sensitive — bypassing cache for a fresh answer.

📍 Route: INTERNET_QUERY  |  Asks for current recommendations about AI tools.
Getting your response from the internet 🌐 ...

🤖 Response (live):
[1] 15 best AI apps I can't live without in 2026 (free + paid)
    Looking for the best AI apps? Here are my top recommendations (for different use cases) based on testing 70 different AI tools over the past 3 years.
    Source: https://www.gumloop.com/blog/best-ai-apps

[2] The best AI productivity tools in 2026
    AI orchestration and automation (Zapier). Chatbots (ChatGPT, Claude, Meta AI). AI agent builders (Zapier, Botpress).
    Source: https://zapier.com/blog/best-ai-productivity-tools/

[3] The Best AI Tools for 2026
    Without a doubt, ChatGPT, Gemini, and Claude are the best AI tools to date. They can provide answers to your everyday questions, do web searches, help with ...
    Source: https://medium.com/artificial-corner/the-best

In [9]:
# Test 5: Time-sensitive query — stock price, always fetched live
result = agentic_rag_with_cache("What is the current stock price of Apple?", cache)

👤 Query: What is the current stock price of Apple?

⏰ Time-sensitive — bypassing cache for a fresh answer.

📍 Route: INTERNET_QUERY  |  Requests real-time stock market data.
Getting your response from the internet 🌐 ...

🤖 Response (live):
[1] AAPL Apple Inc.
    Recent developments for Apple (AAPL) include unusual trading activity in long-dated put options indicating bullish sentiment, and plans to re-enter the ...
    Source: https://finance.yahoo.com/quote/AAPL/

[2] Stock Price
    Stock Quote: NASDAQ: AAPL ; Day's Open332.53 ; Closing Price332.41 ; Volume36.0 ; Intraday High335.48 ; Intraday Low330.70.
    Source: https://investor.apple.com/stock-price/default.aspx

[3] Buy or Sell Apple Stock - AAPL Stock Price Quote & News
    At a current price of $330.87, the stock is +0.1% higher than the low and still -1.4% under the high. Trading activity shows a volume of 35.98M, compared to an ...
    Source: https://robinhood.com/us/en/stocks/AAPL/

[4] AAPL: Apple Inc. - Stock Price, Qu

In [10]:
# Test 6: Cache MISS — INTERNET_QUERY, stored after Ares API call
result = agentic_rag_with_cache("What are the most popular open-source LLMs?", cache)

👤 Query: What are the most popular open-source LLMs?

❌ Cache MISS — running Agentic RAG pipeline...

📍 Route: INTERNET_QUERY  |  Asks about general popularity of open-source LLMs
Getting your response from the internet 🌐 ...

💾 Cached for future similar queries.

🤖 Response:
[1] The best open-source large language models (LLMs)
    We evaluated nine of the best open-source models: DeepSeek V4 Pro 0813, Gemma 4, GLM-5.3, GPT OSS 120B, Kimi K3, MiniMax M3, Nemotron 3 Ultra, ...
    Source: https://www.baseten.co/blog/the-best-open-source-large-language-models-llms/

[2] Best Open Source LLM Leaderboard 2026
    The definitive ranking of every major open source model — compared across reasoning, coding, math, software engineering, and instruction ...
    Source: https://onyx.app/open-llm-leaderboard

[3] Best Open-Source LLMs (Updated July 2026): Top Models
    Which Is the Best Multilingual Open-Source LLM? · Mistral Small 4 for Enterprise Deployment · Gemma 4 for Multilingual Edge Appl

In [11]:
# Test 7: Cache HIT — similar to Test 6
result = agentic_rag_with_cache("Which open-source large language models are most widely used?", cache)

👤 Query: Which open-source large language models are most widely used?

❌ Cache MISS — running Agentic RAG pipeline...

📍 Route: INTERNET_QUERY  |  Asks about general popularity of open-source large language models.
Getting your response from the internet 🌐 ...

💾 Cached for future similar queries.

🤖 Response:
[1] The best open-source large language models (LLMs)
    We evaluated nine of the best open-source models: DeepSeek V4 Pro 0813, Gemma 4, GLM-5.3, GPT OSS 120B, Kimi K3, MiniMax M3, ...
    Source: https://www.baseten.co/blog/the-best-open-source-large-language-models-llms/

[2] Top 7 open source LLMs for 2026
    Unlike proprietary models developed by companies like OpenAI and Google, open source LLMs are licensed to be freely used, modified, and distributed by anyone.
    Source: https://www.instaclustr.com/education/open-source-ai/top-7-open-source-llms-for-2026/

[3] Best Open-source AI models? : r/LocalLLM
    I know its kinda a broad question but i wanted to learn from th

## 6. Role-Based Access Control (RBAC)

So far, any query could reach any knowledge source. In a real deployment, different
people should see different documents — an engineer probably shouldn't be able to pull
up finance's 10-K analysis, and a finance analyst has no business reading internal
on-call runbooks.

This section adds a thin **RBAC (Role-Based Access Control)** layer in front of the
retrieval pipeline: every request is checked against the requesting user's role
*before* anything is embedded, searched, or sent to an LLM. If the role doesn't have
permission, the request is rejected right there — the underlying vector store and
cache are never touched.

**Setup for this demo — 2 roles, 3 files:**

| File | `engineer` | `finance_analyst` |
|---|---|---|
| 📘 OpenAI Agents Documentation (shared) | ✅ | ✅ |
| 📗 Uber 2021 10-K Filing (finance-only) | ❌ | ✅ |
| 🛠️ Internal On-Call Runbook (engineering-only) | ✅ | ❌ |

The OpenAI docs and the 10-K filing reuse the two Qdrant collections already loaded
by `init_rag()`. The on-call runbook is a small mock document defined inline below —
just to show a third, engineering-only knowledge source without having to build and
embed a new Qdrant collection for it.


In [12]:
# ── File registry ────────────────────────────────────────────────────────────
# Each file says where its content actually lives: one of the two existing
# Qdrant collections, or (for the runbook) a small inline mock document.
FILES = {
    "openai_docs": {
        "title": "OpenAI Agents Documentation",
        "source": "qdrant",
        "action": "OPENAI_QUERY",        # matches rag_helpers' collection routing
    },
    "uber_10k": {
        "title": "Uber 2021 10-K Filing",
        "source": "qdrant",
        "action": "10K_DOCUMENT_QUERY",  # matches rag_helpers' collection routing
    },
    "internal_runbook": {
        "title": "Internal Engineering On-Call Runbook",
        "source": "mock",
        "content": """
            ON-CALL RUNBOOK — Payments Service
            1. Page the on-call engineer via PagerDuty if the error rate exceeds
               2% for more than 5 minutes.
            2. Open the #incidents Slack channel and start a war room if it's a P1.
            3. Roll back the most recent deploy with `deploy rollback payments-service`.
            4. A postmortem is required for any P1 or P2 incident within 48 hours.
        """,
    },
}

# ── Roles → the files each role is allowed to read ─────────────────────────
ROLE_PERMISSIONS = {
    "engineer":        {"openai_docs", "internal_runbook"},
    "finance_analyst": {"openai_docs", "uber_10k"},
}

# ── Users → role ─────────────────────────────────────────────────────────────
# Stand-in for a real identity provider (SSO/JWT claims, an internal users
# table, etc.) — swap this out for a real lookup in production.
USERS = {
    "alice": "engineer",
    "bob":   "finance_analyst",
}


def has_access(user_id: str, file_id: str) -> bool:
    """True only if the user's role is explicitly permitted to read this file."""
    role = USERS.get(user_id)
    return role is not None and file_id in ROLE_PERMISSIONS.get(role, set())


In [13]:
import asyncio
import rag_helpers  # gives us access to the shared OpenAI client + internal retriever


def _answer_from_mock_doc(user_query: str, doc_text: str) -> str:
    """Grounded answer generated from a small local document — no Qdrant needed."""
    prompt = f"""
    Based ONLY on the following internal document, answer the user's question.
    If the document doesn't contain the answer, say so explicitly.

    Document:
    {doc_text}

    Question: {user_query}
    """
    response = rag_helpers._openaiclient.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[{"role": "system", "content": prompt}],
    )
    return response.choices[0].message.content


def secure_agentic_rag(user_id: str, file_id: str, user_query: str) -> str:
    """
    RBAC-gated retrieval.

    The access check happens BEFORE any embedding, vector search, cache lookup,
    or LLM call — an unauthorized request never touches the underlying data,
    it's rejected right here.
    """
    CYAN, RED, GREEN, BOLD, RESET = "\033[96m", "\033[91m", "\033[92m", "\033[1m", "\033[0m"

    role      = USERS.get(user_id, "UNKNOWN")
    file_meta = FILES.get(file_id)

    print(f"{BOLD}{CYAN}👤 User:{RESET} {user_id}  (role: {role})")
    print(f"{BOLD}{CYAN}📄 Requested file:{RESET} {file_meta['title'] if file_meta else file_id}")

    if file_meta is None:
        print(f"{RED}❌ Unknown file: {file_id}{RESET}\n")
        return f"❌ Unknown file: {file_id}"

    if not has_access(user_id, file_id):
        print(f"{RED}🚫 ACCESS DENIED{RESET} — role '{role}' has no permission for this file.\n")
        return (
            f"🚫 Access denied: your role ('{role}') does not have permission "
            f"to view '{file_meta['title']}'."
        )

    print(f"{GREEN}✅ Access granted{RESET} — retrieving...\n")

    if file_meta["source"] == "mock":
        result = _answer_from_mock_doc(user_query, file_meta["content"])
    else:
        result = asyncio.run(rag_helpers._retrieve_and_respond(user_query, file_meta["action"]))

    print(f"{BOLD}{CYAN}🤖 Response:{RESET}\n{result}\n")
    return result


### Demo — same two questions, different roles

Each pair below sends the *same* query as both `alice` (engineer) and `bob`
(finance_analyst), so you can see the identical request get allowed for the file
the role is entitled to and denied for the file it isn't.


In [14]:
print("="*70)
print("1) alice (engineer) asks about the OpenAI docs — SHARED file → ALLOWED")
print("="*70)
secure_agentic_rag("alice", "openai_docs", "How do I build an agent with the OpenAI Agents SDK?")


1) alice (engineer) asks about the OpenAI docs — SHARED file → ALLOWED
👤 User: alice  (role: engineer)
📄 Requested file: OpenAI Agents Documentation
✅ Access granted — retrieving...

🤖 Response:
An agent in the OpenAI Agents SDK is built from three core parts:

1. **Model** — the LLM that reasons and makes decisions  
2. **Tools** — functions or APIs the agent can call  
3. **Instructions** — rules that define the agent’s behavior [1]

## 1. Install and configure the SDK

```bash
pip install openai-agents
```

Set your API key:

```bash
export OPENAI_API_KEY="your-api-key"
```

## 2. Define a tool

Tools let the agent take actions or retrieve information outside the model.

```python
from agents import function_tool

@function_tool
def get_weather(city: str) -> str:
    """Return the current weather for a city."""
    # Replace this with a real weather API call.
    return f"The weather in {city} is sunny and 22°C."
```

## 3. Create the agent

```python
from agents import Agent

weath

'An agent in the OpenAI Agents SDK is built from three core parts:\n\n1. **Model** — the LLM that reasons and makes decisions  \n2. **Tools** — functions or APIs the agent can call  \n3. **Instructions** — rules that define the agent’s behavior [1]\n\n## 1. Install and configure the SDK\n\n```bash\npip install openai-agents\n```\n\nSet your API key:\n\n```bash\nexport OPENAI_API_KEY="your-api-key"\n```\n\n## 2. Define a tool\n\nTools let the agent take actions or retrieve information outside the model.\n\n```python\nfrom agents import function_tool\n\n@function_tool\ndef get_weather(city: str) -> str:\n    """Return the current weather for a city."""\n    # Replace this with a real weather API call.\n    return f"The weather in {city} is sunny and 22°C."\n```\n\n## 3. Create the agent\n\n```python\nfrom agents import Agent\n\nweather_agent = Agent(\n    name="Weather agent",\n    instructions=(\n        "You are a helpful weather assistant. "\n        "Use the get_weather tool when the

In [15]:
print("="*70)
print("2) alice (engineer) asks about the on-call runbook — ENGINEER-ONLY → ALLOWED")
print("="*70)
secure_agentic_rag("alice", "internal_runbook", "What should I do if the error rate spikes?")


2) alice (engineer) asks about the on-call runbook — ENGINEER-ONLY → ALLOWED
👤 User: alice  (role: engineer)
📄 Requested file: Internal Engineering On-Call Runbook
✅ Access granted — retrieving...

🤖 Response:
If the error rate exceeds 2% for more than 5 minutes:

1. Page the on-call engineer via PagerDuty.
2. If it’s a P1, open the `#incidents` Slack channel and start a war room.
3. Roll back the most recent deploy with:
   ```bash
   deploy rollback payments-service
   ```
4. If it’s a P1 or P2 incident, complete a postmortem within 48 hours.



'If the error rate exceeds 2% for more than 5 minutes:\n\n1. Page the on-call engineer via PagerDuty.\n2. If it’s a P1, open the `#incidents` Slack channel and start a war room.\n3. Roll back the most recent deploy with:\n   ```bash\n   deploy rollback payments-service\n   ```\n4. If it’s a P1 or P2 incident, complete a postmortem within 48 hours.'

In [16]:
print("="*70)
print("3) alice (engineer) asks about the Uber 10-K — FINANCE-ONLY → DENIED")
print("="*70)
secure_agentic_rag("alice", "uber_10k", "What was Uber's revenue in 2021?")


3) alice (engineer) asks about the Uber 10-K — FINANCE-ONLY → DENIED
👤 User: alice  (role: engineer)
📄 Requested file: Uber 2021 10-K Filing
🚫 ACCESS DENIED — role 'engineer' has no permission for this file.



"🚫 Access denied: your role ('engineer') does not have permission to view 'Uber 2021 10-K Filing'."

In [17]:
print("="*70)
print("4) bob (finance_analyst) asks about the Uber 10-K — FINANCE-ONLY → ALLOWED")
print("="*70)
secure_agentic_rag("bob", "uber_10k", "What was Uber's revenue in 2021?")


4) bob (finance_analyst) asks about the Uber 10-K — FINANCE-ONLY → ALLOWED
👤 User: bob  (role: finance_analyst)
📄 Requested file: Uber 2021 10-K Filing
✅ Access granted — retrieving...

🤖 Response:
Uber's revenue in 2021 was **$3,208,323,000**, or approximately **$3.21 billion**. [1]



"Uber's revenue in 2021 was **$3,208,323,000**, or approximately **$3.21 billion**. [1]"

In [18]:
print("="*70)
print("5) bob (finance_analyst) asks about the OpenAI docs — SHARED file → ALLOWED")
print("="*70)
secure_agentic_rag("bob", "openai_docs", "How do I build an agent with the OpenAI Agents SDK?")


5) bob (finance_analyst) asks about the OpenAI docs — SHARED file → ALLOWED
👤 User: bob  (role: finance_analyst)
📄 Requested file: OpenAI Agents Documentation
✅ Access granted — retrieving...

🤖 Response:
To build an agent with the OpenAI Agents SDK, define three core components:

1. **Model** — the language model responsible for reasoning and decision-making  
2. **Tools** — functions or APIs the agent can call  
3. **Instructions** — rules that define the agent’s behavior [1]

### 1. Install and configure the SDK

```bash
pip install openai-agents
```

Set your API key:

```bash
export OPENAI_API_KEY="your-api-key"
```

### 2. Define a tool

Tools let the agent interact with external systems or perform actions.

```python
from agents import Agent, Runner, function_tool

@function_tool
def get_weather(city: str) -> str:
    """Return the current weather for a city."""
    # Replace this with a real weather API call.
    return f"The weather in {city} is sunny and 22°C."
```

### 3. Cr

'To build an agent with the OpenAI Agents SDK, define three core components:\n\n1. **Model** — the language model responsible for reasoning and decision-making  \n2. **Tools** — functions or APIs the agent can call  \n3. **Instructions** — rules that define the agent’s behavior [1]\n\n### 1. Install and configure the SDK\n\n```bash\npip install openai-agents\n```\n\nSet your API key:\n\n```bash\nexport OPENAI_API_KEY="your-api-key"\n```\n\n### 2. Define a tool\n\nTools let the agent interact with external systems or perform actions.\n\n```python\nfrom agents import Agent, Runner, function_tool\n\n@function_tool\ndef get_weather(city: str) -> str:\n    """Return the current weather for a city."""\n    # Replace this with a real weather API call.\n    return f"The weather in {city} is sunny and 22°C."\n```\n\n### 3. Create the agent\n\n```python\nweather_agent = Agent(\n    name="Weather agent",\n    instructions=(\n        "You are a helpful weather assistant. "\n        "Use the get_we

In [19]:
print("="*70)
print("6) bob (finance_analyst) asks about the on-call runbook — ENGINEER-ONLY → DENIED")
print("="*70)
secure_agentic_rag("bob", "internal_runbook", "What should I do if the error rate spikes?")


6) bob (finance_analyst) asks about the on-call runbook — ENGINEER-ONLY → DENIED
👤 User: bob  (role: finance_analyst)
📄 Requested file: Internal Engineering On-Call Runbook
🚫 ACCESS DENIED — role 'finance_analyst' has no permission for this file.



"🚫 Access denied: your role ('finance_analyst') does not have permission to view 'Internal Engineering On-Call Runbook'."

**Notes on taking this further:**

- This is an *allow-list* model checked in application code. A production system
  would typically pull the user's role from an identity provider (SSO/JWT claims)
  rather than a hardcoded `USERS` dict.
- For collections that mix content different roles can *partially* see, push the
  permission check into the vector store itself using **payload-based filters**
  (Qdrant supports this natively) so restricted chunks never even enter the
  retrieved context — rather than filtering whole files in/out like this demo does.
  The `10k_data` collection is already labelled for this: every point carries
  `metadata.company` (`"Lyft, Inc."` / `"Uber Technologies, Inc."`) and most carry
  `metadata.fiscal_year`. So a role restricted to Lyft filings becomes a filter, not
  an if-statement:

  ```python
  from qdrant_client import models

  hits = await _qdrant.query_points(
      collection_name="10k_data",
      query=embedding,
      limit=3,
      query_filter=models.Filter(must=[
          models.FieldCondition(key="metadata.company",
                                match=models.MatchValue(value="Lyft, Inc.")),
      ]),
  )
  ```

  The difference matters: an if-statement decides what to *show* after retrieval ran,
  while a filter decides what retrieval is even allowed to see. Only the second one
  survives an LLM that summarises its context.
- Every check above is a natural point to log an audit trail (who asked for what,
  and whether it was allowed) — useful for compliance in regulated environments.


## 7. Inspect the Cache

View all entries currently stored in the semantic cache.

In [20]:
print(f"Total cached entries: {len(cache.cache['questions'])}")
print(f"FAISS index size: {cache.index.ntotal}\n")

for i, (q, a) in enumerate(zip(cache.cache['questions'], cache.cache['response_text'])):
    print(f"[{i}] Q: {q}")
    print(f"    A: {a[:120]}...\n" if len(a) > 120 else f"    A: {a}\n")

Total cached entries: 4
FAISS index size: 4

[0] Q: What was Uber's revenue in 2021?
    A: Uber's revenue in 2021 was **$3,208,323,000**, or approximately **$3.21 billion**.[1]

[1] Q: How do I build an agent with the OpenAI Agents SDK?
    A: An agent in the OpenAI Agents SDK is built from three core parts:

1. **Model** — the LLM that reasons and makes decisio...

[2] Q: What are the most popular open-source LLMs?
    A: [1] The best open-source large language models (LLMs)
    We evaluated nine of the best open-source models: DeepSeek V4 ...

[3] Q: Which open-source large language models are most widely used?
    A: [1] The best open-source large language models (LLMs)
    We evaluated nine of the best open-source models: DeepSeek V4 ...



## Assignment: Extend the System

Try one or more of these extensions:

1. **Adjustable similarity threshold** — Experiment with `threshold=0.1` (stricter) vs `threshold=0.35` (looser). How does it affect hit rate and answer quality?

2. **Cache TTL (Time-To-Live)** — Add an expiry timestamp to each cache entry. Stale entries (e.g., older than 7 days) should be evicted and re-fetched.

3. **Sub-query division** — Before checking the cache, use a GPT call to split compound questions (e.g., *"What was Uber and Lyft revenue in 2021?"*) into sub-queries. Check and populate the cache per sub-query.

4. **Cache analytics** — Track and display cache hit rate, average latency for hits vs misses, and the most-queried topics over a session.